# Intervention B — ColdLLM-Style Synthetic Interaction Generation for Cold-Start Items

This notebook sets up **Intervention B**: using an LLM as a behavior simulator to generate
synthetic interactions for strict cold-start items, following the two-stage funnel from
ColdLLM (arXiv:2402.09176) — **Filtering Simulation** narrows every cold item's candidate
pool from all users down to the top-K most content-similar, then **Refining Simulation** asks
an LLM a yes/no for each surviving (item, user) pair. Positive predictions become synthetic
interactions, added to `ref_train` before fitting ALS — giving a cold item something to learn
from *before* it has accumulated a single real interaction, which is exactly the case
Section 7 of the baseline notebook shows ALS's own fold-in cannot help with at all
(`recalculate_item`'s closed-form solve returns the exact zero vector at k=0). That
synthetic-augmented ALS fit is then wrapped in CBHCF (Section 6), so the comparison against
the baseline isolates the synthetic interactions' contribution rather than conflating it with
an ALS-vs-CBHCF architecture difference.

**Tests two Refining-stage prompting strategies in one run**: **B1 (direct)** asks the bare
yes/no question with choice-constrained decoding; **B2 (reasoning)** asks the LLM to give a
one-sentence justification *before* answering, on the hypothesis that a brief chain-of-thought
surfaces content-relevance signal a forced-immediate answer skips past. Everything upstream of
Refining -- dataset, baseline, item metadata, and Filtering Simulation (Section 4) -- is
identical for both and computed only **once**; the two strategies only fork at Refining
(Section 5) and the ALS/CBHCF fit that depends on it (Section 6), then rejoin for a
side-by-side comparison (Section 7). Both strategies live in `recsys/coldllm.py` behind a
single `reasoning` flag threaded through `_build_prompt`/`yes_probability`/`refine_candidates`.

**Compares CBHCF+ColdLLM against CBHCF** — `steel_thread.ipynb`'s content-based hybrid CF is
the strongest baseline in that notebook, so Intervention B's synthetic interactions are
layered onto CBHCF itself (wrapping the same synthetic-augmented ALS fit CBHCF would wrap
around any collaborative model), not compared against a different model family. That isolates
what the synthetic interactions actually contribute, holding the content term -- and its
lambda -- fixed at exactly what `steel_thread.ipynb` tuned. The baseline CBHCF curve itself is
**loaded from that notebook's persisted `../results/baseline_cf_*.pkl`, never recomputed** --
see Section 2; only its content space and lambda are reused to build the CBHCF+ColdLLM variants
in Section 6.

**Reuses the baseline notebook's dataset, split, and cache conventions** (`../data/cache/`,
`cache_pickle`, the same `DATA_PATH`/`SPLIT_PARAMS`) so its cached artifacts and the Amazon
Books item metadata are directly comparable, and — since ALS fitting is cached by
`(dataset fingerprint, ALS params, seed)` — reusing the exact same `ALS_PARAMS` here means any
matching ALS fit loads from disk instantly if `steel_thread.ipynb` has already run on this
machine, rather than refitting from scratch.

## 0. Environment & imports

In [ ]:
%%time
import platform
import sys
import time

sys.path.insert(0, "..")  # notebook lives in notebooks/; the recsys/ package is one level up

import numpy as np
import scipy.sparse as sparse
import matplotlib
try:
    get_ipython()            # defined only under IPython/Jupyter -> keep the inline backend
except NameError:
    matplotlib.use("Agg")    # plain `python script.py`: headless, save figures to PNG instead
import matplotlib.pyplot as plt
import implicit
import wandb

from recsys import load
from recsys import cf
from recsys import cbhcf
from recsys import content
from recsys import coldllm
from recsys import eval as ev

DEVICE = "cuda:1"   # same GPU convention as steel_thread.ipynb -- GPU 1 is idle; GPU 0
                    # is shared with the desktop. ALS fit() is CPU-only regardless.

print(f"python     {platform.python_version()}  ({platform.machine()})")
print(f"implicit   {implicit.__version__}")

# Weights & Biases (W&B) -- experiment tracking, same convention as the baseline notebook.
WANDB_ENTITY_NAME = "mads-team-cold-start"
WANDB_PROJECT_NAME = "llm-cold-start-recsys"
RUN_TAG = time.strftime("%Y%m%d-%H%M%S")

WANDB_ENABLED = False  # flip to True to actually upload runs to W&B (off by default; disabled = no-op)
WANDB_MODE = "online" if WANDB_ENABLED else "disabled"

## 1. Load the same dataset/split as the baseline notebook

Identical `DATA_PATH`/`SPLIT_PARAMS`/`cache_pickle` pattern as `steel_thread.ipynb` -- same
dataset fingerprint `FP`, so the two notebooks' cached artifacts (the Dataset itself, ALS fits
keyed on `(FP, ALS_PARAMS, seed)`, the content matrix keyed on `FP`) are shared, not
duplicated.

In [ ]:
%%time
# Identical cache_pickle helper to steel_thread.ipynb -- see that notebook's Section 0 markdown
# for the fingerprinting rationale (keyed on inputs for the Dataset itself, on outputs for
# everything downstream). Duplicated here rather than imported so this notebook stays runnable on
# its own; keep the two definitions in sync if either changes.
import os
import pickle
import hashlib


def cache_pickle(name, compute, params=None):
    tag = "" if params is None else "_" + hashlib.md5(repr(params).encode()).hexdigest()[:8]
    path = f"../data/cache/{name}{tag}.pkl"
    if os.path.exists(path):
        with open(path, "rb") as f:
            return pickle.load(f)
    val = compute()
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "wb") as f:
        pickle.dump(val, f, protocol=pickle.HIGHEST_PROTOCOL)
    return val


DATA_PATH = "../data/filtered/books_5core_common.parquet"
SPLIT_PARAMS = dict(cold_item_fraction=0.10, cold_val_fraction=0.10)

dataset = cache_pickle("books_dataset", lambda: load.load_dataset(data_path=DATA_PATH, **SPLIT_PARAMS),
                       params=load.load_params_fingerprint(DATA_PATH, **SPLIT_PARAMS))
FP = load.dataset_fingerprint(dataset)   # every downstream cache key includes this

N_REVEAL = 20
K_LEVELS = list(range(N_REVEAL + 1))

print(f"users: {dataset.n_users:,}   items: {dataset.n_items:,}")
print(f"cold items (test): {len(dataset.cold_item_ids):,}")
print(f"ref_train non-zeros: {dataset.ref_train.nnz:,}   ref_test non-zeros: {dataset.ref_test.nnz:,}")
print(f"reserved cold-item test interactions: {dataset.test_matrix.nnz:,}")
print(f"dataset fingerprint: {FP}")

## 2. Baseline — CBHCF, loaded from cached results

Intervention B is only interesting relative to the *best* baseline, so this notebook compares
against **CBHCF** (`steel_thread.ipynb`'s content-based hybrid CF), not Popularity or plain ALS.
CBHCF is expensive to fit (a full `N_SEEDS`-averaged sweep, ~30 min on GPU), so instead it loads the
already-computed `cbhcf_curve`/`cbhcf_ref` straight out of that notebook's persisted
`../results/baseline_cf_*.pkl`. If that file doesn't exist yet, run `steel_thread.ipynb` (with
`RUN_CBHCF = True`) once first.

`ALS_PARAMS` is still defined below — not for a baseline fit, but because Section 6 fits a
fresh ALS model per prompting strategy on `ref_train + synthetic_matrix`, and reuses these exact
params so those fits are directly comparable to the plain ALS fits elsewhere in the project.

In [ ]:
%%time
import glob

# ALS_PARAMS is only used later (Section 6) to fit the ColdLLM-augmented models -- it is NOT a
# baseline fit here. Kept identical to steel_thread.ipynb's so those fits are comparable.
N_FACTORS = 64
N_ITERATIONS = 20
ALS_PARAMS = dict(factors=N_FACTORS, regularization=0.01, iterations=N_ITERATIONS)

# Load CBHCF's cached curve/reference from steel_thread.ipynb's persisted results -- never
# recomputed here. Filenames are timestamp-suffixed, so sorted()[-1] is the most recent run.
baseline_result_files = sorted(glob.glob("../results/baseline_cf_*.pkl"))
if not baseline_result_files:
    raise FileNotFoundError(
        "No ../results/baseline_cf_*.pkl found. Run steel_thread.ipynb (with RUN_CBHCF = True) "
        "at least once first -- Intervention B compares against its cached CBHCF curve rather "
        "than recomputing CBHCF here."
    )
baseline_results_path = baseline_result_files[-1]
with open(baseline_results_path, "rb") as f:
    baseline_results = pickle.load(f)

if not baseline_results["config"].get("RUN_CBHCF"):
    raise ValueError(
        f"{baseline_results_path} was saved with RUN_CBHCF=False, so it has no cbhcf_curve. "
        f"Rerun steel_thread.ipynb with CBHCF enabled."
    )
if list(baseline_results["config"]["dataset_fingerprint"]) != list(FP):
    print(f"WARNING: {baseline_results_path}'s dataset fingerprint doesn't match this "
          f"notebook's FP -- CBHCF was evaluated on a different train/test split. Rerun "
          f"steel_thread.ipynb on the current data before trusting the comparison below.")

cbhcf_curve = baseline_results["mode_a"]["cbhcf_curve"]
cbhcf_ref = baseline_results["reference"]["cbhcf"]
print(f"Loaded CBHCF baseline from {baseline_results_path} "
      f"(lambda={baseline_results['config']['CBHCF_LAMBDA']}, "
      f"saved {baseline_results['config']['timestamp']}).")
print("CBHCF within-item ceiling (all pre-test history revealed): "
      + "  ".join(f"{m}={cbhcf_ref['mean'][m]:.4f}" for m in cbhcf_ref["mean"]))

## 3. Item content: plain text for LLM prompts, TF-IDF vectors for Filtering

Same metadata pipeline `steel_thread.ipynb`'s CBHCF section (5c) uses --
`content.load_item_documents` against the Amazon Books metadata parquet, `content.BOOKS_FIELD_MAP`
for the role -> column mapping (title, creator, taxonomy, blurb, reviews). Two representations
are built from it, for two different consumers:

- `item_metadata` -- one **plain readable string per item**, the five role arrays flattened and
  space-joined. Feeds the Refining Simulation prompt (Section 5) and the leakage/overlap
  diagnostics -- an LLM prompt needs a string, not a sparse vector.
- `item_content` -- the row-L2-normalized TF-IDF matrix `content.ContentSpace.transform` builds
  (`n_items x n_terms`, cosine similarity via inner product). Feeds Filtering Simulation
  (Section 4) and the CBHCF wrap (Section 6) -- built **once** here and reused by both, cached
  under the exact same `cache_pickle` key `steel_thread.ipynb` uses, so it loads that notebook's
  already-computed matrix directly instead of rebuilding it.

In [ ]:
%%time
# Cached by dataset fingerprint alone (the metadata file and field map are fixed)
def _item_metadata():
    docs = content.load_item_documents(
        "../data/filtered/books_meta_5core_common.parquet", dataset,
        field_map=content.BOOKS_FIELD_MAP)
    return [
        " ".join(filter(None, (docs[role][i] for role in content.ROLES)))
        for i in range(dataset.n_items)
    ]


item_metadata = cache_pickle("coldllm_item_metadata", _item_metadata, params=(FP,))
n_empty = sum(1 for t in item_metadata if not t.strip())
n_empty_cold = sum(1 for i in dataset.cold_item_ids if not item_metadata[i].strip())
print(f"item_metadata: {len(item_metadata):,} items, {n_empty:,} with no text at all "
      f"(of which cold: {n_empty_cold:,})")
print(f"\nexample (cold item_index {int(dataset.cold_item_ids[0])}):")
print(f"  {item_metadata[int(dataset.cold_item_ids[0])][:300]}")

# TF-IDF content vectors -- the SAME content space steel_thread.ipynb's CBHCF section builds.
# Filtering Simulation (Section 4) ranks candidate users by cosine similarity against this;
# Section 6 reuses this exact object for the CBHCF wrap.
CBHCF_FIELD_WEIGHTS = None  # None -> content.DEFAULT_WEIGHTS -- must match steel_thread.ipynb's
warm_item_ids = np.unique(dataset.ref_train.nonzero()[1])


def _content_matrix():
    docs = content.load_item_documents(
        "../data/filtered/books_meta_5core_common.parquet", dataset,
        field_map=content.BOOKS_FIELD_MAP)
    space = content.fit_content_space(docs, warm_item_ids, weights=CBHCF_FIELD_WEIGHTS, min_df=2)
    return space.transform(docs)


item_content = cache_pickle("books_content_tfidf", _content_matrix,
                            params=(FP, "bm25f", "warmfit", "mindf2", repr(CBHCF_FIELD_WEIGHTS)))
print(f"item_content: {item_content.shape} ({item_content.nnz:,} nonzeros)")

## 4. Filtering Simulation (TF-IDF content similarity)

Stage 1 doesn't touch an LLM at all: `coldllm.user_content_profile` builds each user's content
profile the same way CBHCF's own frozen user profile is built (`_row_scaled` applied to
`ref_train`, projected through `item_content` from Section 3), then `coldllm.filter_candidates`
ranks users per cold item by inner product against that profile. Both `item_content` and the
profiles are L2-normalized rows, so the inner product IS cosine similarity -- no GPU/vLLM
engine needed for this stage.

In [ ]:
%%time
COLDLLM_TOP_K = 50  # Stage 1: candidate users kept per cold item

coldllm_cold_items = dataset.cold_item_ids  # always the full cold-item population
print(f"Running Filtering Simulation on the full cold-item population: "
      f"{len(coldllm_cold_items):,} items (top_k={COLDLLM_TOP_K}).")

# TF-IDF cosine similarity
# Filtering's candidates are independent of the Refining prompting strategy, so this runs
# exactly once and is reused by every strategy in Section 5.
user_profiles = coldllm.user_content_profile(dataset.ref_train, item_content)
candidates = coldllm.filter_candidates(item_content, user_profiles, coldllm_cold_items,
                                       top_k=COLDLLM_TOP_K)
print(f"user_profiles: {user_profiles.shape} ({user_profiles.nnz:,} nonzeros)")

## 5. Refining Simulation — both prompting strategies

This is the one stage that differs between **B1 (direct)** and **B2 (reasoning)** --
see `recsys/coldllm.py`'s `_build_prompt`/`yes_probability` for exactly how the prompt and the
decoding constraint change. `STRATEGIES` below is looped over: each iteration builds its own `VLLMColdLLMSimulator` and calls `refine_candidates(...,
reasoning=...)` against the SAME `candidates` from Section 4, caching the result keyed by
strategy (among the other Filtering/threshold params) so B1 and B2 never collide on the same
cache file.

**Refining cost**: one LLM chat call per (item, candidate_user) pair that survives Filtering --
up to `COLDLLM_TOP_K` calls per cold item, run over the full cold-item population, **for each
strategy**. B1's completions are capped at 5 tokens (just the yes/no token); B2's are capped at
200 (room for a one-sentence justification plus the final `Answer: yes`/`Answer: no` line) --
so expect B2's pass to take meaningfully longer even though the call count is identical.

In [ ]:
%%time
COLDLLM_MODEL = "Qwen/Qwen3-14B"
# Other options worth trying -- this model does the yes/no judgment below, so "good at simple
# yes/no classification" matters more here than general chat quality:
#   "Qwen/Qwen2.5-7B-Instruct" -- half the params of the 14B above, so meaningfully cheaper on
#                                 GPU memory; worth dropping back to if the 14B's cost isn't
#                                 justified by an accuracy difference on this task.
#   "meta-llama/Llama-3.1-8B-Instruct" -- vLLM's most mature/heavily-tested architecture, so the
#                                        safest fallback if either of the others hits a rough edge;
#                                        strong general judgment, just edged out by Qwen above on
#                                        instruction-following/classification-style benchmarks.
#   "microsoft/Phi-3.5-mini-instruct" -- only ~3.8B params, so meaningfully cheaper still;
#                                        Microsoft's Phi line is trained specifically for strong
#                                        reasoning-per-parameter, which a binary yes/no judgment
#                                        doesn't need much of -- worth trying first if GPU memory
#                                        or throughput is the binding constraint.
COLDLLM_THRESHOLD = 0.5  # cutoff for keeping a synthetic interaction (hard 0.0/1.0 predictions
                          # here, so this just needs to sit strictly between them)

STRATEGIES = ["direct", "reasoning"]  # B1, B2 -- see recsys/coldllm.py's `reasoning` flag

synthetic_matrices = {}
for strategy in STRATEGIES:
    reasoning = strategy == "reasoning"

    def _refine(reasoning=reasoning):
        simulator = coldllm.VLLMColdLLMSimulator(model=COLDLLM_MODEL)
        return coldllm.refine_candidates(simulator, candidates, item_metadata, dataset.ref_train,
                                         threshold=COLDLLM_THRESHOLD, reasoning=reasoning)

    synthetic_matrix = cache_pickle(
        "coldllm_synthetic",
        _refine,
        params=(FP, COLDLLM_MODEL, COLDLLM_TOP_K, COLDLLM_THRESHOLD, strategy),
    )
    synthetic_matrices[strategy] = synthetic_matrix

    # Diagnostics -- see the notebook intro and recsys/coldllm.py's docstring for why both matter.
    items_with_synthetic = np.unique(synthetic_matrix.nonzero()[1])
    overlap = synthetic_matrix.multiply(dataset.test_matrix)
    n_overlap = int(overlap.nnz)
    print(f"[{strategy}] synthetic interactions: {synthetic_matrix.nnz:,}   "
          f"cold items covered: {len(items_with_synthetic):,} of {len(coldllm_cold_items):,}   "
          f"test-set overlap: {n_overlap:,} ({n_overlap / max(synthetic_matrix.nnz, 1):.2%})")

## 6. Warm-up curve — does synthetic augmentation help CBHCF, and does either prompting strategy help more?

Per strategy: ALS is fit on `ref_train + synthetic_matrices[strategy]`, then wrapped in
`cbhcf.CBHCFModel`, reusing `item_content` (Section 3) and the cached `CBHCF_LAMBDA` -- so the
only thing that can differ from `cbhcf_curve`, or between the two strategies, is the wrapped
collaborative model's synthetic augmentation. The content score cache is built once (first
strategy) and borrowed by the second via `reuse_from`, since it doesn't depend on the
collaborative model at all (see `CBHCFModel.build_content_cache`'s docstring).

`coldllm.SyntheticAugmentedDataset` makes `fold_in()` see the synthetic interactions at every
`k`, including `k=0` -- where `fold_in`'s closed-form solve would otherwise return the exact
zero vector. It overrides **only** `revealed_item_users_at_k`, never `revealed_matrix_at_k` --
the latter is what `eval.py` uses to build the real-only "already interacted" exclusion matrix,
and synthetic interactions aren't guaranteed disjoint from `dataset.test_matrix` the way real
revealed history is, so folding them in there would risk masking out the very test items being
measured.

Every curve here -- cached `cbhcf_curve` and each strategy's `curves[strategy]` -- covers the
same cold-item population, content term, and lambda, so the comparison is controlled at every
`k`.

In [ ]:
%%time
K = 100
METRICS = ev.METRICS  # ["NDCG", "Precision", "Recall", "HitRate", "AUC"]

CBHCF_LAMBDA = baseline_results["config"]["CBHCF_LAMBDA"]

# item_content (Section 3) is already the exact TF-IDF matrix CBHCF needs -- built once there,
# reused here for both strategies' CBHCFModel.fit.
eval_users_all = np.flatnonzero(np.diff(dataset.test_matrix.tocsr().indptr))

# The content score cache (independent of lambda, seed, and synthetic data -- see class docstring)
# is keyed identically to steel_thread.ipynb's, so this loads its cached file directly instead of
# rebuilding it, as long as that notebook has already run on this machine.
cbhcf_content_cache_path = f"../data/cache/cbhcf_content_{hashlib.md5(repr(FP).encode()).hexdigest()[:8]}.pkl"

In [ ]:
%%time
coldllm_models = {}
cbhcf_coldllm_models = {}
curves = {}

for i, strategy in enumerate(STRATEGIES):
    fit_matrix = dataset.ref_train + synthetic_matrices[strategy]
    coldllm_model = cf.ALSModel(random_state=0, **ALS_PARAMS).fit(fit_matrix)
    coldllm_model.prepare_gpu_recommend(dataset, candidates="warm_cold", device=DEVICE)
    coldllm_models[strategy] = coldllm_model

    cbhcf_model = cbhcf.CBHCFModel(content_weight=CBHCF_LAMBDA).fit(
        dataset.ref_train, item_content=item_content, cf_model=coldllm_model)
    cbhcf_model.prepare_gpu_recommend()
    if i == 0:
        # First strategy: load from steel_thread.ipynb's cache file if present, else build fresh.
        cbhcf_model.build_content_cache(
            eval_users_all, device=DEVICE, gpu_device=DEVICE, path=cbhcf_content_cache_path)
    else:
        # Second strategy onward: borrow the already-built cache in memory -- no recompute, no
        # disk reload (see cbhcf.CBHCFModel.build_content_cache's `reuse_from` docstring).
        cbhcf_model.build_content_cache(
            eval_users_all, reuse_from=cbhcf_coldllm_models[STRATEGIES[0]])
    cbhcf_model.calibrate(eval_users_all)
    cbhcf_coldllm_models[strategy] = cbhcf_model
    print(f"[{strategy}] Fit ALS on ref_train + synthetic_matrix ({fit_matrix.nnz:,} nonzeros); "
          f"CBHCF+ColdLLM lambda={CBHCF_LAMBDA}  s_cf={cbhcf_model._s_cf:.5f}  s_cb={cbhcf_model._s_cb:.5f}")

    augmented_dataset = coldllm.SyntheticAugmentedDataset(
        dataset, synthetic_matrices[strategy], restrict_to_item_ids=coldllm_cold_items)
    curve, n_eval_per_k = ev.sweep([cbhcf_model], augmented_dataset, K_LEVELS, K=K)
    curves[strategy] = curve
    print(f"[{strategy}] warm-up sweep (k=0):  "
          + "  ".join(f"{m}={curve[m]['mean'][0]:.4f}" for m in METRICS))

## 7. Compare prompting strategies

Both strategies' curves, plus the cached CBHCF baseline, side by side. A positive delta below
means the reasoning strategy (B2) scored higher than the direct strategy (B1) at that metric.

In [ ]:
%%time
print("B2 (reasoning) vs. B1 (direct) -- per-metric delta at k=0 and averaged across the full "
      "warm-up curve (positive = reasoning strategy scores higher):")
for m in METRICS:
    direct_mean = np.array(curves["direct"][m]["mean"])
    reasoning_mean = np.array(curves["reasoning"][m]["mean"])
    delta_k0 = float(reasoning_mean[0] - direct_mean[0])
    delta_avg = float(np.mean(reasoning_mean - direct_mean))
    print(f"  {m:<10} k=0: {delta_k0:+.4f}   avg over curve: {delta_avg:+.4f}")

In [ ]:
%%time
fig, axes = plt.subplots(2, 3, figsize=(12, 8), sharex="col")
axes = axes.flatten()

STRATEGY_STYLE = {
    "direct": dict(color="#2980b9", label="CBHCF+ColdLLM B1 (direct)"),
    "reasoning": dict(color="#27ae60", label="CBHCF+ColdLLM B2 (reasoning)"),
}

for i, (ax, metric) in enumerate(zip(axes, METRICS)):
    show_label = (lambda text: text) if i == 0 else (lambda text: None)
    ax.plot(K_LEVELS, cbhcf_curve[metric]["mean"], marker="o", markersize=4, linewidth=1.5,
            color="#c0392b", label=show_label("CBHCF (cached)"))
    for strategy in STRATEGIES:
        style = STRATEGY_STYLE[strategy]
        ax.plot(K_LEVELS, curves[strategy][metric]["mean"], marker="o", markersize=4, linewidth=1.5,
                color=style["color"], label=show_label(style["label"]))
    ax.set_title("AUC" if metric == "AUC" else f"{metric}@{K}")
    ax.set_xlabel("k -- real interactions revealed per cold item")
    ax.grid(axis="y", alpha=0.2)

axes[0].set_ylabel("score")
axes[0].legend(fontsize=8, loc="upper left")
fig.suptitle(f"Warm-up curve -- CBHCF baseline vs. B1 (direct) vs. B2 (reasoning), "
             f"model={COLDLLM_MODEL}, full {len(dataset.cold_item_ids):,}-item cold population",
             y=1.04)
plt.tight_layout()
plt.show()

## 8. Persist results

In [ ]:
%%time
import json, datetime

results = {
    "config": {
        "ALS_PARAMS": ALS_PARAMS, "K": K, "DEVICE": DEVICE,
        "COLDLLM_MODEL": COLDLLM_MODEL, "COLDLLM_TOP_K": COLDLLM_TOP_K,
        "COLDLLM_THRESHOLD": COLDLLM_THRESHOLD, "STRATEGIES": STRATEGIES,
        "n_cold_items_attempted": int(len(coldllm_cold_items)),
        "n_synthetic_interactions": {s: int(synthetic_matrices[s].nnz) for s in STRATEGIES},
        "dataset_fingerprint": list(FP),
        "cbhcf_baseline_source": baseline_results_path,
        "cbhcf_lambda": CBHCF_LAMBDA,
        "timestamp": datetime.datetime.now().isoformat(timespec="seconds"),
    },
    "cold_start_k0": {
        "cbhcf_baseline": {m: cbhcf_curve[m]["mean"][0] for m in METRICS},
        **{f"cbhcf_coldllm_{s}": {m: curves[s][m]["mean"][0] for m in METRICS} for s in STRATEGIES},
    },
    "warm_up_curve": {
        "cbhcf_baseline": cbhcf_curve,
        **{f"cbhcf_coldllm_{s}": curves[s] for s in STRATEGIES},
        "n_eval_per_k": n_eval_per_k, "k_levels": K_LEVELS,
    },
}

os.makedirs("../results", exist_ok=True)
stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
base = f"../results/intervention_b_coldllm_{stamp}"
with open(base + ".pkl", "wb") as f:
    pickle.dump(results, f)
with open(base + ".json", "w") as f:
    json.dump(results, f, indent=2, default=lambda o: float(o) if hasattr(o, "__float__") else str(o))
print(f"Saved results to {base}.pkl and {base}.json")

## 9. Summary

- **Objective**: check whether ColdLLM-style synthetic interactions give a strict cold-start
  item something to learn from before it has accumulated a single real interaction -- the exact
  gap Section 7 of `steel_thread.ipynb` shows ALS's own fold-in cannot close at `k=0` -- measured
  as **CBHCF+ColdLLM against CBHCF itself** (Section 6), isolating the intervention's marginal
  contribution rather than conflating it with an ALS-vs-CBHCF architecture difference, across
  the whole warm-up curve, not just `k=0`.
- **Prompting strategies compared**: B1 (direct yes/no) vs. B2 (one-sentence reason, then
  yes/no) -- Section 7 reports the per-metric delta, at `k=0` and averaged across the whole
  warm-up curve, between the two. A positive delta means the reasoning strategy scored higher.
- **Cost tradeoff**: B2's Refining completions are capped at 200 tokens vs. B1's 5, so Section 5
  takes meaningfully longer for B2 than for B1 given the same call count -- worth weighing
  against whatever improvement (if any) Section 7 finds before adopting B2 as the default
  strategy.